<a href="https://colab.research.google.com/github/deniwidi/data-science-2026/blob/main/Pertemuan10_Deni_Widi_Alfian_240401010340.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Pertemuan ke-10

- Nama    : Deni Widi Alfian
- NIM     : 240401010340
- Kelas   : IF405

## Langkah 1: Muat dan Eksplorasi Data
Baca dataset, periksa tipe data, dan hitung proporsi kelas churn untuk memastikan dataset bersifat
imbalanced.

In [7]:
import pandas as pd

# Membaca dataset
df = pd.read_csv("telco_churn.csv")

# Periksa tipe data dan ukuran dataset
print("Ukuran dataset:", df.shape)

# Hitung proporsi kelas churn untuk memastikan dataset bersifat imbalanced
print("\nProporsi Target (Churn):")
print(df["Churn"].value_counts(normalize=True))

Ukuran dataset: (7043, 21)

Proporsi Target (Churn):
Churn
No     0.73463
Yes    0.26537
Name: proportion, dtype: float64


## Langkah 2: Preprocessing
Lakukan encoding pada fitur kategorikal menggunakan `pd.get_dummies()` atau `LabelEncoder`,
lalu bagi data menjadi latih/uji secara stratified untuk menjaga proporsi kelas:


In [16]:
from sklearn.model_selection import train_test_split

# Membersihkan data (opsional: menangani nilai kosong pada TotalCharges jika ada spasi)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()

# TODO: memisahkan X (fitur) dan y (target = Churn)
X = df.drop("Churn", axis=1)
# Mengubah target menjadi numerik (1 untuk Yes, 0 untuk No)
y = df["Churn"].apply(lambda x: 1 if x == 'Yes' else 0)

# TODO: encoding fitur kategorikal (mis. pd.get_dummies)
X = pd.get_dummies(X, drop_first=True)

# Membagi data uji dan latih dengan menjaga proporsi kelas (stratify)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Jumlah data latih:", len(X_tr))
print("Jumlah data uji:", len(X_te))

Jumlah data latih: 5625
Jumlah data uji: 1407


## Langkah 3: Latih Model
Bangun `RandomForestClassifier` dengan `class_weight="balanced"` untuk menangani
imbalance:

In [17]:
from sklearn.ensemble import RandomForestClassifier

# Membangun RandomForestClassifier dengan parameter sesuai modul
rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42
)

# Melatih model dengan data latih
rf.fit(X_tr, y_tr)
print("Model Random Forest berhasil dilatih.")

Model Random Forest berhasil dilatih.


## Langkah 4: Evaluasi
Hitung precision, recall, F1-score, dan ROC-AUC dengan fokus pada kelas churn (kelas 1):


In [18]:
from sklearn.metrics import classification_report, roc_auc_score

# TODO: hitung prediksi dan probabilitas
pred = rf.predict(X_te)
proba = rf.predict_proba(X_te)[:, 1] # Probabilitas khusus untuk kelas positif (Churn)

# TODO: tampilkan classification report dan ROC-AUC
print("Classification Report:")
print(classification_report(y_te, pred))

print("ROC-AUC Score:", round(roc_auc_score(y_te, proba), 4))

Classification Report:
              precision    recall  f1-score   support

           0       0.83      0.89      0.86      1033
           1       0.63      0.51      0.56       374

    accuracy                           0.79      1407
   macro avg       0.73      0.70      0.71      1407
weighted avg       0.78      0.79      0.78      1407

ROC-AUC Score: 0.8309


## Langkah 5: Prediksi Probabilitas dan Simpulkan
Gunakan predict_proba untuk menghasilkan probabilitas churn tiap pelanggan, lalu tulis 3–4
kalimat kesimpulan mengenai temuan utama dari model.

In [19]:
# TODO: hitung probabilitas churn (predict_proba)
# (Probabilitas sudah dihitung pada variabel 'proba' di atas)
print("Contoh Probabilitas Churn untuk 5 Pelanggan Pertama di Data Uji:")
for i, p in enumerate(proba[:5]):
    print(f"Pelanggan {i+1}: {p*100:.2f}% risiko churn")

Contoh Probabilitas Churn untuk 5 Pelanggan Pertama di Data Uji:
Pelanggan 1: 2.00% risiko churn
Pelanggan 2: 73.33% risiko churn
Pelanggan 3: 1.00% risiko churn
Pelanggan 4: 9.33% risiko churn
Pelanggan 5: 16.33% risiko churn


## Kesimpulan


*   Penyesuaian bobot kelas `(class_weight="balanced")` terbukti sukses membuat model lebih sensitif dalam mendeteksi pelanggan yang berisiko tinggi.

*   Model Random Forest berhasil mengukur persentase risiko churn (berhenti berlangganan) pada masing-masing pelanggan.

*   Tim retensi dapat langsung memprioritaskan tindakan pencegahan secara akurat pada pelanggan yang paling rentan pergi melalui probabilitas churn yang dihasilkan.

